# Session 11 · Introduction to Apache Spark

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Diploma-in-Data-Science/blob/main/term-2/02-big-data-systems-with-hadoop/code/session-11-introduction-to-apache-spark.ipynb)

Diploma in Data Science · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run the cells from top to bottom. Every example is self-contained, so you can also run any single cell on its own.

**Setup:** the Spark examples need PySpark (and Java, which Google Colab already has).

In [1]:
%pip install -q pyspark

### Example 1 · Word count with RDDs

In [2]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder.master("local[2]").appName("rdd")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext

lines = sc.parallelize([
    "spark keeps data in memory", "hadoop writes data to disk",
    "spark runs on yarn", "yarn manages the hadoop cluster"], numSlices=2)
counts = (lines.flatMap(lambda l: l.split())       # narrow
               .map(lambda w: (w, 1))              # narrow
               .reduceByKey(lambda a, b: a + b))   # wide -> shuffle
print("partitions:", lines.getNumPartitions())
print("top words :", counts.sortBy(lambda x: (-x[1], x[0])).take(5))
import re
print("\nlineage:")
print(re.sub(r" at \S+:\d+ \[\]", "", counts.toDebugString().decode()))

partitions: 2


top words : [('data', 2), ('hadoop', 2), ('spark', 2), ('yarn', 2), ('cluster', 1)]

lineage:
(2) PythonRDD[12] at RDD
 |  MapPartitionsRDD[4] at mapPartitions
 |  ShuffledRDD[3] at partitionBy
 +-(2) PairwiseRDD[2] at reduceByKey
    |  PythonRDD[1] at reduceByKey
    |  ParallelCollectionRDD[0] at readRDDFromFile


### Example 2 · Lazy evaluation

In [3]:
import time
from pyspark.sql import SparkSession
spark = (SparkSession.builder.master("local[2]").appName("lazy")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext

t0 = time.time()
rdd = sc.parallelize(range(1, 2_000_001), 4)
squares = rdd.map(lambda x: x * x).filter(lambda x: x % 3 == 0)    # transformations only
print(f"after transformations : {time.time() - t0:.3f} s (nothing has run yet)")
print("count =", squares.count())                                   # action -> job runs now
print("the action triggered a job: map + filter ran over all 4 partitions")
print("first 5 =", squares.take(5))

after transformations : 0.007 s (nothing has run yet)


count = 666666
the action triggered a job: map + filter ran over all 4 partitions
first 5 = [9, 36, 81, 144, 225]


### Example 3 · DataFrames, SQL and the Catalyst plan

In [4]:
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.master("local[2]").appName("df")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

df = spark.createDataFrame([
    ("Ali", "CS", 3.4, 2023), ("Sara", "DS", 3.9, 2022), ("Omar", "CS", 2.8, 2023),
    ("Hina", "DS", 3.6, 2023), ("Zain", "SE", 3.1, 2022), ("Iqra", "CS", 3.7, 2022)],
    "name string, dept string, cgpa double, batch int")
df.printSchema()

result = (df.filter(F.col("cgpa") >= 3.0)
            .groupBy("dept").agg(F.count("*").alias("students"), F.round(F.avg("cgpa"), 2).alias("avg_cgpa"))
            .orderBy("dept"))
result.show()

df.createOrReplaceTempView("students")
spark.sql("SELECT batch, MAX(cgpa) AS best FROM students GROUP BY batch ORDER BY batch").show()
result.explain()

root
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- cgpa: double (nullable = true)
 |-- batch: integer (nullable = true)



+----+--------+--------+
|dept|students|avg_cgpa|
+----+--------+--------+
|  CS|       2|    3.55|
|  DS|       2|    3.75|
|  SE|       1|     3.1|
+----+--------+--------+



+-----+----+
|batch|best|
+-----+----+
| 2022| 3.9|
| 2023| 3.6|
+-----+----+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [dept#1 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(dept#1 ASC NULLS FIRST, 4), ENSURE_REQUIREMENTS, [plan_id=145]
      +- HashAggregate(keys=[dept#1], functions=[count(1), avg(cgpa#2)])
         +- Exchange hashpartitioning(dept#1, 4), ENSURE_REQUIREMENTS, [plan_id=142]
            +- HashAggregate(keys=[dept#1], functions=[partial_count(1), partial_avg(cgpa#2)])
               +- Project [dept#1, cgpa#2]
                  +- Filter (isnotnull(cgpa#2) AND (cgpa#2 >= 3.0))
                     +- Scan ExistingRDD[name#0,dept#1,cgpa#2,batch#3]




### Example 4 · Jobs, stages and caching

In [5]:
import time
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.master("local[2]").appName("cache")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

big = (spark.range(0, 3_000_000)
       .withColumn("city", (F.col("id") % 5).cast("string"))
       .withColumn("amount", F.sqrt(F.col("id")) * 3))
clean = big.filter(F.col("amount") > 100).withColumn("tax", F.col("amount") * 0.17)

def timed(label, fn):
    t0 = time.time(); out = fn(); print(f"{label:<28} {time.time() - t0:6.2f} s  -> {out}")

timed("count, not cached", lambda: clean.count())
clean.cache()
timed("count, first (fills cache)", lambda: clean.count())
timed("count, from cache", lambda: clean.count())
timed("group by city, from cache", lambda: clean.groupBy("city").count().count())
print("storage level:", clean.storageLevel)
clean.unpersist()

count, not cached              0.68 s  -> 2998888


count, first (fills cache)     3.60 s  -> 2998888


count, from cache              0.33 s  -> 2998888


group by city, from cache      1.50 s  -> 5
storage level: Disk Memory Deserialized 1x Replicated


DataFrame[id: bigint, city: string, amount: double, tax: double]